# LAB REPORT 02
## Exploratory Data Analysis (EDA) & Digital Conversion Funnel Analysis Using Python

**Course Name:** Marketing Analytics  
**Course Code:** DS 4115  
**Lab Number:** 02  

**Submitted By:** [Your Name]  
**Student ID:** [Your Student ID]  
**Section:** [Your Section]  
**Submitted To:** [Faculty Name]  
**Department:** [Department Name]  
**University:** United International University  
**Submission Date:** [Date]

---

### Report Overview

This notebook combines the practical Python implementation and the detailed academic lab report. It generates a synthetic digital-session dataset, analyzes a three-stage conversion funnel, cleans missing data, performs correlation analysis, and documents the findings.

**Important:** Replace the student-information placeholders above before submission. Run all code cells from top to bottom so that the notebook contains fresh outputs and visualizations.

# 1. Introduction

Digital platforms generate large amounts of behavioral data through user interactions, including website visits, product views, add-to-cart activities, and purchases. Analyzing this data helps businesses understand customer behavior and identify potential problems in the conversion process.

In this laboratory exercise, Python is used to generate synthetic GA4-style session logs and perform Exploratory Data Analysis (EDA). A three-step digital conversion funnel is constructed to analyze user progression from website visits to purchases.

Data-cleaning techniques are also applied to handle missing values, and correlation analysis is performed to investigate relationships between user-engagement variables and purchase behavior.

# 2. Objectives

The main objectives of this laboratory exercise are:

1. To generate a synthetic digital telemetry dataset containing 5,000 user sessions.
2. To construct a three-step digital conversion funnel.
3. To calculate conversion rates at different funnel stages.
4. To identify and handle missing values using pandas.
5. To perform Exploratory Data Analysis (EDA).
6. To generate and interpret a correlation heatmap using Seaborn.
7. To identify potential conversion drop-off points and derive business insights.

# 3. Tools and Technologies

| Tool / Library | Purpose |
|---|---|
| Google Colab / Jupyter Notebook | Python development environment |
| Python | Data analysis and processing |
| Pandas | DataFrame creation, inspection, and cleaning |
| NumPy | Numerical operations and synthetic data generation |
| Matplotlib | Data visualization |
| Seaborn | Correlation heatmap generation |

# 4. Methodology

The practical workflow consists of five main stages:

1. Environment setup and library import.
2. Synthetic dataset generation and inspection.
3. Conversion funnel construction and metric calculation.
4. Missing-value detection and data cleaning.
5. Correlation analysis, visualization, and interpretation.

The dataset is synthetic. Therefore, the results demonstrate the analysis workflow and simulated patterns rather than actual customer behavior.

## 4.1 Environment Setup

The required Python libraries are imported below. Seaborn's white-grid theme is configured to improve the readability of visualizations.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visualization style
sns.set_theme(style="whitegrid")

print("Environment successfully set up!")

## 4.2 Synthetic Dataset Generation

A synthetic dataset containing 5,000 digital sessions is generated using NumPy and Pandas.

The dataset includes the following attributes:

| Column | Description |
|---|---|
| `session_id` | Unique identifier for each session |
| `user_id` | Identifier of the website visitor |
| `device_category` | Device used: desktop, mobile, or tablet |
| `session_start_flag` | Indicates whether a session started |
| `add_to_cart_flag` | Indicates whether a product was added to the cart |
| `purchase_flag` | Indicates whether a purchase occurred |
| `page_views` | Number of pages viewed during a session |
| `time_on_site_sec` | Time spent on the website in seconds |

The simulation uses a 30% probability of adding a product to the cart. For sessions with an add-to-cart event, a 60% purchase probability is used. Fifty missing values are intentionally introduced in `time_on_site_sec` to practice data cleaning.

A fixed random seed (`42`) is used to make the generated data reproducible.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)
n_rows = 5000

# Generate session attributes
session_ids = [f"sess_{i:04d}" for i in range(1, n_rows + 1)]
user_ids = [f"user_{np.random.randint(1000, 2000)}" for _ in range(n_rows)]

device_categories = np.random.choice(
    ['desktop', 'mobile', 'tablet'],
    size=n_rows,
    p=[0.5, 0.4, 0.1]
)

# Generate funnel flags
session_start_flag = np.ones(n_rows, dtype=int)

add_to_cart_flag = np.random.choice(
    [0, 1],
    size=n_rows,
    p=[0.7, 0.3]
)

purchase_flag = np.where(
    add_to_cart_flag == 1,
    np.random.choice([0, 1], size=n_rows, p=[0.4, 0.6]),
    0
)

# Generate engagement metrics
page_views = np.random.poisson(lam=3, size=n_rows) + 1
time_on_site_sec = page_views * np.random.randint(20, 60, size=n_rows)

# Create DataFrame
df = pd.DataFrame({
    'session_id': session_ids,
    'user_id': user_ids,
    'device_category': device_categories,
    'session_start_flag': session_start_flag,
    'add_to_cart_flag': add_to_cart_flag,
    'purchase_flag': purchase_flag,
    'page_views': page_views,
    'time_on_site_sec': time_on_site_sec
})

# Introduce 50 artificial missing values
df.loc[
    np.random.choice(n_rows, 50),
    'time_on_site_sec'
] = np.nan

# Save the original dataset
df.to_csv('ga4_sample_session_logs.csv', index=False)

print("Dataset 'ga4_sample_session_logs.csv' successfully generated!")
display(df.head())

## 4.3 Dataset Inspection

The dataset structure is inspected using `df.info()`. This displays the number of records, column names, non-null counts, data types, and memory usage.

The expected initial structure is 5,000 records and 8 columns. The `time_on_site_sec` column contains 50 missing values, while the other columns should contain 5,000 non-null values.

In [ ]:
# Inspect dataset structure
df.info()

# Confirm dataset dimensions
print("Dataset shape:", df.shape)

# 5. Results and Discussion

## 5.1 Digital Conversion Funnel Analysis

A conversion funnel tracks user progression through key milestones. In this lab, the funnel consists of three stages:

1. **Session Start:** A visitor starts a website session.
2. **Add to Cart:** The visitor adds a product to the cart.
3. **Purchase:** The visitor completes a purchase.

The number of unique sessions at each stage is calculated. Conversion rates are measured relative to the total number of sessions:

\[
\text{Conversion Rate (\%)} =
\frac{\text{Sessions at Current Stage}}{\text{Total Sessions}} \times 100
\]

In [ ]:
# Calculate funnel metrics
total_sessions = df['session_id'].nunique()

sessions_with_cart = df[
    df['add_to_cart_flag'] == 1
]['session_id'].nunique()

sessions_with_purchase = df[
    df['purchase_flag'] == 1
]['session_id'].nunique()

# Create funnel summary
funnel_data = {
    'Funnel Step': [
        '1. Session Start',
        '2. Add to Cart',
        '3. Purchase'
    ],
    'User Sessions': [
        total_sessions,
        sessions_with_cart,
        sessions_with_purchase
    ]
}

funnel_df = pd.DataFrame(funnel_data)

# Calculate conversion rate relative to session start
funnel_df['Conversion Rate (%)'] = (
    funnel_df['User Sessions'] / total_sessions
) * 100

display(funnel_df)

### Funnel Results and Interpretation

The generated dataset is expected to produce the following funnel results:

| Funnel Stage | User Sessions | Conversion Rate |
|---|---:|---:|
| Session Start | 5,000 | 100.0% |
| Add to Cart | 1,470 | 29.4% |
| Purchase | 905 | 18.1% |

Out of 5,000 website sessions, 1,470 sessions resulted in adding products to the cart. This represents a 29.4% conversion rate from session start to add-to-cart.

A total of 905 sessions resulted in purchases, representing an overall purchase conversion rate of 18.1%.

The results indicate that a significant proportion of visitors did not proceed to the add-to-cart stage.

## 5.2 Cart Conversion and Abandonment Analysis

The cart-to-purchase conversion rate measures the proportion of cart sessions that resulted in a purchase.

\[
\text{Cart-to-Purchase Rate (\%)} =
\frac{\text{Purchase Sessions}}{\text{Cart Sessions}} \times 100
\]

Cart abandonment rate is calculated as:

\[
\text{Cart Abandonment Rate (\%)} =
100 - \text{Cart-to-Purchase Rate}
\]

In [ ]:
# Calculate cart conversion and abandonment rates
cart_conversion = (
    sessions_with_purchase / sessions_with_cart
) * 100

cart_abandonment = 100 - cart_conversion

print(f"Cart-to-Purchase Rate: {cart_conversion:.2f}%")
print(f"Cart Abandonment Rate: {cart_abandonment:.2f}%")

### Cart Analysis Interpretation

The cart-to-purchase conversion rate is **61.56%**, while the cart abandonment rate is **38.44%**.

This means that approximately 61.56% of sessions that added products to the cart resulted in purchases, while 38.44% did not complete the purchase.

This indicates a potential opportunity to improve the checkout process and investigate possible purchasing barriers. The synthetic dataset does not contain checkout-step or payment-failure information, so the specific causes of abandonment cannot be identified from this analysis alone.

## 5.3 Missing-Value Analysis and Data Cleaning

Real-world telemetry data may contain missing or inconsistent records. Missing values are first identified using `isnull().sum()`.

The initial dataset contains 50 missing values in `time_on_site_sec`.

In [ ]:
# Detect missing values in each column
print(df.isnull().sum())

### Median Imputation

Median imputation is used to replace missing values in `time_on_site_sec`.

The median is the middle value of a sorted dataset. It is often preferred over the mean when data may contain extreme values because it is less sensitive to outliers.

Additional cleaning operations are performed:

- Rows with missing `session_id` values are removed.
- Values in `device_category` are converted to lowercase.
- Leading and trailing whitespace is removed from categorical values.

In [ ]:
# Calculate median and impute missing time values
median_time = df['time_on_site_sec'].median()
df['time_on_site_sec'] = df['time_on_site_sec'].fillna(median_time)

# Drop rows with missing session IDs, if any
df = df.dropna(subset=['session_id'])

# Standardize categorical values
df['device_category'] = (
    df['device_category'].str.lower().str.strip()
)

# Verify remaining missing values
print(
    "Data cleaning completed. Missing values remaining:",
    df.isnull().sum().sum()
)

### Data-Cleaning Result

Before cleaning, 50 missing values were present in `time_on_site_sec`. After median imputation and the additional cleaning operations, the dataset contains **zero remaining missing values**.

The cleaned dataset is retained in the DataFrame `df` for subsequent analysis.

## 5.4 Correlation Analysis

Correlation analysis is used to examine the strength and direction of linear relationships between numerical variables.

The correlation coefficient ranges from -1 to +1:

| Correlation Value | Interpretation |
|---|---|
| +1 | Perfect positive linear correlation |
| 0 | No linear correlation |
| -1 | Perfect negative linear correlation |

The constant `session_start_flag` column is excluded from the correlation analysis because all its values are 1. A constant variable has zero variance, so its correlation with other variables is undefined.

The selected variables are:

- `add_to_cart_flag`
- `purchase_flag`
- `page_views`
- `time_on_site_sec`

In [ ]:
# Select numerical variables with variation
numeric_cols = [
    'add_to_cart_flag',
    'purchase_flag',
    'page_views',
    'time_on_site_sec'
]

# Generate correlation matrix
corr_matrix = df[numeric_cols].corr()

display(corr_matrix)

### Correlation Matrix Results

The key correlation values from the generated dataset are:

| Variable Pair | Correlation |
|---|---:|
| Add to Cart and Purchase | 0.728 |
| Page Views and Time on Site | 0.804 |
| Page Views and Purchase | -0.001 |
| Time on Site and Purchase | 0.004 |

The correlation between add-to-cart activity and purchase behavior is strongly positive (0.728). Page views and time spent on the website also show a strong positive correlation (0.804).

However, page views and time spent on the website have almost zero linear correlation with purchase behavior in this synthetic dataset.

## 5.5 Correlation Heatmap

A Seaborn heatmap is generated to visualize the correlation matrix. The annotations display the correlation coefficients, while the color scale represents the direction and strength of the relationships.

In [ ]:
# Plot the correlation heatmap
plt.figure(figsize=(9, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1,
    center=0
)

plt.title(
    "Digital Behavioral Correlation Heatmap",
    fontsize=14
)

plt.tight_layout()
plt.show()

### Heatmap Interpretation

The heatmap highlights two notable relationships:

1. **Add to Cart and Purchase (0.73):** A strong positive relationship is observed. This is expected because the synthetic data-generation process allows purchases only for sessions that have an add-to-cart event.
2. **Page Views and Time on Site (0.80):** A strong positive relationship indicates that sessions with more page views generally have longer browsing durations in the generated data.

The correlations between page views and purchase (-0.001), and between time on site and purchase (0.004), are close to zero. This means that these variables do not show a meaningful linear relationship with purchase behavior in this particular synthetic dataset.

Correlation does not establish causation. These results should be interpreted as simulated statistical patterns, not proof that one behavior causes another.

# 6. Business Insights

Based on the conversion funnel and correlation analysis, the following insights can be identified:

1. **Visitor Drop-off:** Only 29.4% of sessions resulted in adding products to the cart. This suggests a potential opportunity to improve product discovery, product presentation, and the user experience. The analysis does not identify the exact reason for the drop-off.

2. **Cart Abandonment:** Approximately 38.44% of cart sessions did not result in purchases. A business could investigate checkout usability, shipping costs, payment options, and other possible barriers. These are recommendations for further investigation, not causes established by this dataset.

3. **User Engagement:** Page views and time spent on the website have a strong positive correlation (0.804), suggesting that more extensive browsing is associated with longer sessions in the simulated data.

4. **Purchase Behavior:** Add-to-cart activity has a strong positive correlation with purchase behavior (0.728). This is consistent with the funnel logic used to generate the synthetic data.

5. **Synthetic Data Limitation:** Because the dataset is generated using predefined probabilities, the observed relationships may not represent real customer behavior. Real-world validation would require actual digital analytics data.

# 7. Conclusion

In this laboratory exercise, Exploratory Data Analysis (EDA) and Digital Conversion Funnel Analysis were performed using Python.

A synthetic dataset containing 5,000 digital sessions was generated to simulate e-commerce user behavior. The conversion funnel analysis showed that 29.4% of sessions resulted in adding products to the cart, while 18.1% resulted in purchases.

Data cleaning was performed using median imputation to handle 50 missing values in the `time_on_site_sec` column. Categorical data was also standardized using pandas. After cleaning, no missing values remained.

The correlation analysis revealed a strong positive relationship between add-to-cart and purchase behavior (0.728). Page views and time spent on the website also showed a strong positive correlation (0.804), while their individual correlations with purchase were nearly zero.

Overall, this lab demonstrated how Python-based data analysis can help understand simulated customer behavior, measure conversion performance, identify funnel drop-offs, and support data-driven marketing decisions. Since the dataset is synthetic, the findings should be interpreted as simulated patterns rather than real-world causal relationships.

# 8. Appendix: Exporting the Datasets

The original dataset and the cleaned dataset are saved as CSV files. The original file preserves the intentionally introduced missing values, while the cleaned file contains the imputed and standardized data.

In [ ]:
# Save the cleaned dataset
df.to_csv('ga4_cleaned_session_logs.csv', index=False)

print("Original dataset: ga4_sample_session_logs.csv")
print("Cleaned dataset: ga4_cleaned_session_logs.csv")
print("Notebook: Lab2_Conversion_Funnel_EDA_Detailed_Report.ipynb")

---

## Submission Checklist

Before submitting, verify the following:

- [ ] Replace all student-information placeholders on the cover page.
- [ ] Run all code cells from top to bottom.
- [ ] Confirm that the funnel table and heatmap are displayed.
- [ ] Confirm that the missing-value check returns zero.
- [ ] Confirm that both CSV files are generated.
- [ ] Save or download the completed `.ipynb` notebook.
- [ ] Push the completed notebook to your individual student GitHub repository.

**End of Lab Report 02**